# 03 · Aim 1: are the synergies stable and real?

**Owner:** Person C · **Read first:** Tresch, Cheung & d'Avella 2006, *Matrix factorization algorithms for the identification of muscle synergies* (J Neurophysiol)

Low-dimensional EMG can come from things that aren't synergies: chance, or signal leaking between
neighbouring electrodes. Each check below gives **one number and one figure**. Use k from notebook 02
(start with 4 until it's chosen).

## 🎯 Goal and milestones
*Draft milestones: agree them as a team on Friday, then edit here.*


**Proposal:** **Check 0 (stability)** is ✅ **Aim 1**: the proposal picks k by held-out error *and* stability across fits.
**Checks 1 to 4 are ➕ beyond the proposal.** They are optional extras, marked below.

**Goal:** show the k synergies from notebook 02 are stable, and (extras) not an artifact.

**Why:** NMF will *always* return k synergies, even from noise. Stability is the minimum the proposal needs.
The extra checks rule out chance and signal leaking between neighbouring electrodes, which makes the bottleneck
easier to defend if a reviewer asks.

**Milestones**
1. ✅ Stability: 10 random starts give the same synergies
2. ➕ Shuffle test: real data beats channel-shuffled data
3. ➕ New-movement test: synergies fitted on 12 tasks still explain the other 3
4. ➕ Neighbour test: synergies spread across the band, not 2 to 3 adjacent electrodes
5. ➕ Two-hands test: left and right synergies match (cosine similarity)

**Done when:** the stability result (required), plus a pass/fail table for any extras you ran.

In [ ]:
# ── Setup: works on Colab and locally ─────────────────────────────────────────
# After this cell the working directory is the repo root, wherever you run from.
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ or Path("/content").exists()
if IN_COLAB:
    ROOT = Path("/content/emg2pose")
    if not ROOT.exists():
        !git clone https://github.com/Wilbertbh-Tan/emg2pose.git {ROOT}
    !pip install -q h5py
else:
    ROOT = Path.cwd().resolve()
    while not (ROOT / "setup.py").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    assert (ROOT / "setup.py").exists(), "Run this notebook from inside the emg2pose repo"
os.chdir(ROOT)

PROCESSED = ROOT / "data" / "processed_mini.npz"   # made by notebook 01 (data/ is never committed)

In [ ]:
# ── Load the shared data made by notebook 01 (runs 01 first if needed) ───────
import numpy as np
if not PROCESSED.exists():
    %run notebooks/experiments/01_data_prep/01_data_prep.ipynb
d = np.load(PROCESSED)

emg        = d["emg"]          # (N, 16)  EMG envelope, 50 samples/s, non-negative
pose       = d["pose"]         # (N, 20)  joint angles in degrees
pose_valid = d["pose_valid"]   # (N,)     False where pose tracking failed
rec        = d["rec"]          # (N,)     which recording (0–29) each row comes from
split      = d["split"]        # (N,)     0 = train, 1 = test, -1 = gap (use neither)
stage_of_rec, side_of_rec = d["stage"], d["side"]   # (30,) task name and hand per recording

train, test = split == 0, split == 1
SEED = 0   # pass random_state=SEED to every model so results repeat
print(f"{len(emg)} frames | {train.sum()} train | {test.sum()} test | {len(stage_of_rec)} recordings")

K = 4   # TODO: replace with K_CHOSEN from notebook 02

### The variables you start with
`N` = number of frames in all 30 recordings together (≈ 90,000: 30 recordings × ~60 s × 50 frames/s).
Every row of every array is one frame, 20 ms long. Boolean arrays are **masks**: `emg[train]` keeps only the train rows.

| Name | Shape | dtype | What it is |
|---|---|---|---|
| `emg` | (N, 16) | float32 | EMG envelope, one column per electrode, ≥ 0, roughly 0–1 |
| `pose` | (N, 20) | float32 | joint angles in **degrees**, one column per joint |
| `pose_valid` | (N,) | bool | False where pose tracking failed (don't use those `pose` rows) |
| `rec` | (N,) | int16 | which recording (0–29) the frame comes from |
| `split` | (N,) | int8 | 0 = train, 1 = test, −1 = gap between them (use neither) |
| `train`, `test` | (N,) | bool | `split == 0`, `split == 1` |
| `stage_of_rec` | (30,) | str | task name of each recording; per frame: `stage_of_rec[rec]` → (N,) |
| `side_of_rec` | (30,) | str | `"left"` or `"right"` per recording; per frame: `side_of_rec[rec]` → (N,) |

Letters used in the docstrings below: **T** = number of frames in the array you pass in (any subset of N),
**k** = number of synergies, **C** = number of gesture classes. Run a helper on a tiny made-up array first
(e.g. `np.random.rand(5, 16)`) to check it returns the shape its docstring promises.

## Before you run anything: your prediction

Write what you expect and why (2–3 sentences). Compare at the end. Surprises are the interesting part.

> *Prediction:*

## Helper: compare two sets of synergies
Synergies come out in random order, so pair them up first (the Hungarian algorithm,
`scipy.optimize.linear_sum_assignment`), then take the cosine similarity of each pair (1 = identical).

In [ ]:
import numpy as np
from scipy.optimize import linear_sum_assignment
from sklearn.decomposition import NMF

K = 4   # replace with K_CHOSEN from notebook 02

def fit_synergies(X: np.ndarray, k: int, seed: int) -> np.ndarray:
    """Fit NMF and return its synergies, each scaled to length 1.

    Parameters
    ----------
    X    : (T, 16) float, non-negative EMG (any subset of frames)
    k    : int, number of synergies
    seed : int, NMF random_state

    Returns
    -------
    W : (k, 16) float, row i = synergy i, with np.linalg.norm(W[i]) == 1
    """
    # TODO
    raise NotImplementedError


def vaf(X: np.ndarray, X_hat: np.ndarray) -> float:
    """Variance accounted for, in % (uncentered). X, X_hat: (T, 16). Same as notebook 02."""
    # TODO
    raise NotImplementedError


def match_synergies(W1: np.ndarray, W2: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Pair up two sets of synergies (they come out in random order) and compare each pair.

    Parameters
    ----------
    W1 : (k, 16) float, one set of synergies, rows of length 1
    W2 : (k, 16) float, another set, rows of length 1

    Returns
    -------
    sims  : (k,) float in [0, 1], cosine similarity of each matched pair (1 = identical)
    order : (k,) int, W2[order[i]] is the partner of W1[i]

    Steps: a (k, k) cosine-similarity matrix, then linear_sum_assignment (it minimises, so pass the negative).
    Check: match_synergies(W, W[::-1]) → sims all 1.0
    """
    # TODO
    raise NotImplementedError

## Check 0 · Stable across random starts
Fit NMF 10 times with different `random_state`. Real structure should give the same synergies each time.
**Figure:** matched similarity of each run to the first run. **Pass if** all are close to 1.

In [ ]:
N_RUNS = 10
X_train = emg[train]                          # (T_train, 16)

W_runs: list[np.ndarray] = []                 # length N_RUNS, each (K, 16)
# TODO: fit_synergies(X_train, K, seed) for seed = 0 … 9, append each W

stability = None                              # (N_RUNS − 1, K): match_synergies(W_runs[0], W_runs[r])[0] for r ≥ 1
# TODO: figure: one dot per run and synergy; report the smallest similarity

## ➕ Beyond the proposal · Check 1 · Shuffle test
Shift each channel's training data in time by its own random amount (`np.roll`). Each channel keeps its own
signal, but which muscles activate *together* is destroyed. Refit and compute VAF for k = 1 … 10.
**Figure:** VAF vs k, real (line) vs shuffled (10 shuffles, shown as a band). **Pass if** real is clearly above.

In [ ]:
def shuffle_channels(X: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    """Shift each channel in time by its own random amount, breaking co-activation.

    Parameters
    ----------
    X   : (T, 16) float, EMG
    rng : np.random.default_rng(seed), the source of the random shifts

    Returns
    -------
    X_shuf : (T, 16) float, same values per column as X, but column j is np.roll-ed by its own shift
    """
    # TODO
    raise NotImplementedError


ks = list(range(1, 11))
N_SHUFFLES = 10
vaf_real: list[float] = []                       # length 10: train-fit, test-scored VAF per k (as in notebook 02)
vaf_shuf = np.full((N_SHUFFLES, len(ks)), np.nan)  # (10 shuffles, 10 k values)
# TODO: fill vaf_real; then for each shuffle s, shuffle the TRAIN and TEST EMG and fill row s of vaf_shuf
# TODO: figure: vaf_real as a line, vaf_shuf as a band (min to max over shuffles)

## ➕ Beyond the proposal · Check 2 · New-movement test
Fit synergies on 12 stages, then explain the other 3 stages with those synergies fixed (only c(t) is refit:
`nmf.transform`). Compare with synergies fitted on those 3 stages themselves.
**Figure:** bar per held-out stage, VAF with borrowed synergies vs own synergies. **Pass if** close to 1 as a ratio.

In [ ]:
stage_per_frame = stage_of_rec[rec]          # (N,) str
stages = np.unique(stage_of_rec)             # (15,) str
held_out = stages[:3]                        # TODO: choose 3 stages (and say why); try other choices too
is_held = np.isin(stage_per_frame, held_out) # (N,) bool

# Fit on train frames of the other 12 stages: emg[train & ~is_held]          → W_borrowed (K, 16)
# Then for each held-out stage s, on its test frames emg[test & (stage_per_frame == s)]:
#   borrowed: keep W fixed and only find c(t) (nmf.transform), then VAF
#   own:      fit NMF on that stage's train frames, VAF on its test frames
vaf_borrowed: dict[str, float] = {}          # stage → VAF (%)
vaf_own: dict[str, float] = {}               # stage → VAF (%)
# TODO (tip: keep the fitted NMF object, not only W, so you can call .transform)
# TODO: figure: bar per held-out stage of vaf_borrowed / vaf_own

## ➕ Beyond the proposal · Check 3 · Neighbour (crosstalk) test
The 16 electrodes sit in a ring around the wrist (assumed from the channel order; check this). For each
synergy, what share of its weight sits on its strongest electrode and the two next to it?
If weight were spread evenly that share would be 3/16 ≈ 19%; if it is on one electrode, 100%.
**Figure:** a polar plot per synergy (16 weights around a circle). **Pass if** synergies span several regions.

In [ ]:
def neighbour_share(w: np.ndarray) -> float:
    """Share of a synergy's weight on its strongest electrode and its two ring neighbours.

    Parameters
    ----------
    w : (16,) float, one synergy (non-negative)

    Returns
    -------
    float in [3/16, 1]. ≈ 0.19 = spread evenly; 1.0 = all on three adjacent electrodes.
    The band is a ring: the neighbours of electrode 0 are 15 and 1.
    """
    # TODO
    raise NotImplementedError


W = None                    # (K, 16): TODO fit_synergies on emg[train]
shares = None               # (K,) float: neighbour_share of each row
# TODO: polar plot per synergy: 16 angles around the circle, radius = weight

## ➕ Beyond the proposal · Check 4 · Two-hands test
Fit synergies on left-hand recordings and on right-hand recordings separately, then compare (matched cosine).
The band on the other wrist is mirrored and may sit rotated, so also try the channel order reversed and
rotated, and keep the best. **Do the same search for a null:** random non-negative synergies, so the search
itself doesn't inflate the score.
**Figure:** k × k similarity heatmap (left vs right). **Pass if** real similarity is clearly above the null.

In [ ]:
def channel_variants(W: np.ndarray) -> list[np.ndarray]:
    """Every way the other wrist's band could line up: 16 rotations × (normal, reversed).

    Parameters
    ----------
    W : (k, 16) float, synergies from one hand

    Returns
    -------
    list of 32 arrays, each (k, 16): the same synergies with the 16 columns rotated and/or reversed
    """
    # TODO
    raise NotImplementedError


def best_match(W_left: np.ndarray, W_right: np.ndarray) -> float:
    """Mean matched cosine similarity, using the best of the 32 channel_variants of W_right.

    W_left, W_right : (k, 16). Returns a float in [0, 1].
    """
    # TODO
    raise NotImplementedError


side_per_frame = side_of_rec[rec]              # (N,) str
W_left  = None   # (K, 16): TODO fit_synergies on emg[train & (side_per_frame == "left")]
W_right = None   # (K, 16): the same for "right"
score_real = None                              # float: best_match(W_left, W_right)
score_null: list[float] = []                   # 100 floats: best_match(W_left, random non-negative (K, 16) with rows of length 1)
# TODO: figure: K × K similarity heatmap (left vs best-aligned right), and score_real against the score_null histogram

## Result
| Check | In proposal? | Number | Pass / fail |
|---|---|---|---|
| 0 Stable | ✅ Aim 1 | | |
| 1 Shuffle | ➕ beyond | | |
| 2 New movements | ➕ beyond | | |
| 3 Neighbours | ➕ beyond | | |
| 4 Two hands | ➕ beyond | | |